### Strcutured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing . Langchain supports multiple schema types and methods for enforcing structred output

### Pydantic

Pydantic Models provide the richest feature set with field validations , descriptions and nested structures

In [1]:
import os

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7bf1fc56e3c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7bf1fc56f0e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel): # BaseModel lets you define a schema for your data.
    title:str=Field(description="The tile of the movie") # Field() adds metadata, validation rules, and descriptions.
    year:int =Field(description="The year movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The Rating of the movie")

In [3]:
model_with_struct_output = model.with_structured_output(Movie , include_raw= True) # To see the original LLM unstructred response
model_with_struct_output

{
  raw: _ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7bf1fc56e3c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7bf1fc56f0e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The tile of the movie', 'type': 'string'}, 'year': {'description': 'The year movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The Rating of the movie', 

In [4]:
response =model_with_struct_output.invoke("Give some info about the Movie Titanic")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for information about the movie Titanic. I need to use the Movie function provided. Let me check the required parameters: title, year, director, and rating. I know the title is Titanic. The year it was released was 1997. The director is James Cameron. The rating is probably a number, maybe from IMDb. Let me confirm those details. Yeah, Titanic directed by James Cameron came out in 1997 and has a high rating, around 7.8 on IMDb. So I'll structure the tool call with those parameters.\n", 'tool_calls': [{'id': 'dyp0npz5a', 'function': {'arguments': '{"director":"James Cameron","rating":7.8,"title":"Titanic","year":1997}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 221, 'total_tokens': 389, 'completion_time': 0.286838965, 'completion_tokens_details': {'reasoning_tokens': 120}, 'prompt_time': 0.018071557, 'prompt_tokens_d

### Nested Structure

In [5]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel): # BaseModel lets you define a schema for your data.
    title:str=Field(description="The tile of the movie") # Field() adds metadata, validation rules, and descriptions.
    year:int 
    director:str
    rating:float| None=Field(description="The Rating of the movie")
    casr:list[Actor]
    genres: list[str]

In [6]:
model_with_struct_nested_output = model.with_structured_output(MovieDetails , include_raw= True) # To see the original LLM unstructred response
model_with_struct_nested_output

{
  raw: _ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7bf1fc56e3c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7bf1fc56f0e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDetails', 'description': '', 'parameters': {'properties': {'title': {'description': 'The tile of the movie', 'type': 'string'}, 'year': {'type': 'integer'}, 'director': {'type': 'string'}, 'rating': {'anyOf': [{'type': 'number'}, {'type': 'null'}], 'description': 'The Rating of the movie'}, 'casr': {'items': {'properties': 

In [7]:
response =model_with_struct_nested_output.invoke("Give some info about the Movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for information about the movie Inception. I need to use the MovieDetails function to get the details. Let me check the parameters required for that function. The function requires title, year, director, rating, cast, and genres.\n\nFirst, I should confirm the title is "Inception". The year it was released is 2010. The director is Christopher Nolan. The rating might be from IMDb, which I think is around 8.8. The cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, and others. The genres would be action, thriller, science fiction. Wait, I need to structure the cast as an array of objects with name and role. Let me make sure I have the correct roles for each actor. Also, check if the rating is a number or null. Since Inception is a popular movie, the rating should be available. Let me put this all together into the function call as specified.\n', 'tool_calls': [{'id': '3hh0yx9z5', 'function'

### TypedDict

TypedDict provides a simpler alertnative using Python's builtin typing .
Ideal when :
- No Runtime validation needed
- Faster Performance
- Passing onl internal data

In [8]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with the details"""
    title: Annotated[str, ..., " The title of the movie"]
    year:Annotated[int, ..., "The year movie was released"]
    director:Annotated[str, ..., "he director of the movie"]
    rating:Annotated[float, ..., "The Rating of the movie"]

In [10]:
model_with_typedDcit= model.with_structured_output(MovieDict)
response =model_with_typedDcit.invoke("Give some info about the Movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [11]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict): # B
    title:str=Field(description="The tile of the movie") # Field() adds metadata, validation rules, and descriptions.
    year:int 
    director:str
    rating:float| None=Field(description="The Rating of the movie")
    casr:list[Actor]
    genres: list[str]

In [13]:
model_with_struct_nested_output = model.with_structured_output(MovieDetails , include_raw= True) # To see the original LLM unstructred response
model_with_struct_nested_output

{
  raw: _ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7bf1fc56e3c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7bf1fc56f0e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDetails', 'description': "dict() -> new empty dictionary\ndict(mapping) -> new dictionary initialized from a mapping object's\n    (key, value) pairs\ndict(iterable) -> new dictionary initialized as if via:\n    d = {}\n    for k, v in iterable:\n        d[k] = v\ndict(**kwargs) -> new dictionary initialized with the name=v

In [14]:
response =model_with_struct_nested_output.invoke("Give some info about the Movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for information about the movie Inception. I need to use the MovieDetails function to get the details. Let me check the parameters required for that function. The function needs title, year, director, rating, cast (as casr), and genres. \n\nFirst, I should confirm the title is "Inception". The year it was released is 2010. The director is Christopher Nolan. The rating might be around 8.8 on IMDb. For the cast, the main actors are Leonardo DiCaprio, Joseph Gordon-Levitt, and Ellen Page. Their roles are Dom Cobb, Arthur, and Ariadne respectively. The genres include Science Fiction, Action, and Thriller.\n\nWait, I need to make sure all the parameters are correctly formatted. The cast should be an array of dictionaries with name and role. Genres are an array of strings. The rating is a number. Let me structure the JSON accordingly. Double-check the required fields to ensure nothing is missing. 

In [15]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### Dataclasses

A data class is a class typically containing mainly data, although there aren't really any restrictions . you can create it using @dataclass decorator

In [16]:
from pydantic import BaseModel, Field

from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name: str=Field(description="Name of the Person")
    email: str=Field(description="Email of the Person")
    phone: str=Field(description="Phone number of the Person")

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [19]:
agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo

)

result= agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Extarct the contact info from John Doe, john@example.com, 123-234-1212"
    }]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='123-234-1212')

In [ ]:
# using dataclass instead of Pydantic or TypedDict to get the structured output

In [22]:
from dataclasses import dataclass
@dataclass
class ContactInfo(BaseModel):
    """Contact information of a person"""
    name: str=Field(description="Name of the Person")
    email: str=Field(description="Email of the Person")
    phone: str=Field(description="Phone number of the Person")

In [23]:
agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo

)

result= agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Extarct the contact info from John Doe, john@example.com, 123-234-1212"
    }]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='123-234-1212')